# Competitor impact: who did the new competitor hurt, and did they come back?

**Context.** Lululime is an athleisure store in Vancouver. A competitor opened across the street on **15 March 2024** and targeted younger shoppers with lower prices on staple items. Leadership believes revenue has since recovered thanks to the loyalty program.

**Questions this notebook answers**
1. Does the drop in sales line up with the competitor's opening on 15 March 2024?
2. Which customers (age band, home area) and which product categories were hit hardest?
3. For the hardest-hit group: did they buy less per visit, visit less often, or stop coming?
4. Did they come back in 2025–2026?

### Summary of the answers
- **Sales dropped the day the competitor opened (15 March 2024).** Revenue from 18–24 year-olds was normal from 1–14 March (+2% vs the same days of 2023), then **−50% from 15–31 March**, and 45–54% down from April to August.
- **18–24s took most of the hit.** Mar–Oct 2024 store revenue was 17% ($681K) below Mar–Oct 2023. 18–24s caused **53% of that loss** while making up only 19% of revenue: their revenue fell **45%**.
- **Leggings fell the most (−30%)**, and in every age band. Customers in **Burnaby and Richmond** pulled back most (−25%).
- **The competitor took young customers and their trips, not their basket size.** 18–24s had 27% fewer active customers and 25% fewer visits each, but spent the same per visit ($113).
- **18–24s never came back.** Their revenue is still ~45% below 2023 in 2025 and 2026. Every other age band recovered and grew 15–32%, so the store's recovery comes from customers aged 25 and over.

## Data and definitions

**The data.** Every Lululime transaction from **1 Nov 2022 to 31 Oct 2026**, one row per item line, joined with product details (category, price, cost) and customer details (age band, home area, sign-up channel). The team cleaned and joined it once in shared code (`src/`), so every notebook starts from the same table:
- `load("lines")` returns the cleaned, joined table: one row per item line, 350,998 rows (1,016 duplicate rows removed).
- `load("visits")` returns one row per transaction (a shopping visit); `load("customers")` returns one row per customer.
- `C` holds values the team agreed on, such as key dates (`C.LOYALTY_LAUNCH` = 1 Jun 2024) and row types (`C.PURCHASE` = "Purchase"). Using `C` keeps every notebook consistent.
- **Competitor date:** the competitor opened on **15 March 2024** (`C.COMPETITOR_OPEN_DAY`, called `OPEN_DAY` below). The shared constant `C.COMPETITOR_OPEN` is 1 March 2024, the month it opened. We use that one only where the analysis works in whole months, and treat **March 2024 as the first competitor month**, even though only its second half had the competitor.

**Purchases only.** Each row is a Purchase, a Return, or a repair drop-off/pickup. We use **purchases** because we're measuring how much customers wanted to buy. Returns (about 5% of revenue) happen weeks later and would shift the timing, and repairs are free ($0).

**Terms**
| Term | Meaning |
|---|---|
| Revenue | Quantity × price actually paid (after holiday discounts) |
| Year over year (YoY) | A month compared with the **same month one year earlier**. December is always ~2× a normal month, so comparing December with November would mislead. Comparing December with last December cancels that out |
| Comparison window | **Mar–Oct 2024 vs. Mar–Oct 2023**. It starts in March, the month the competitor opened (its first two weeks were still before the opening, so this slightly understates the drop). It stops in October because the data ends in October 2026, and we want the same window in every year from 2023 to 2026. 2023 is the only year before the competitor that the data covers |
| Mar–May check | The loyalty program launched in June 2024, inside the window. **Mar–May 2024** had the competitor but no loyalty program yet, so we re-run the key comparison on those months to make sure loyalty isn't driving the result |
| Age band | Customer's age when they first shopped (it doesn't update) |
| Active customer | A customer who made at least one purchase in the period |

In [ ]:
import sys
from pathlib import Path

# Make the repo's shared code (src/) importable; this notebook lives in analysis/a_competitor/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.metrics import load      # shared loader for the cleaned tables
from src import config as C       # shared constants: dates, row types, paths

OUT = C.RESULTS_DIR / "a_competitor"     # charts are saved here
OUT.mkdir(parents=True, exist_ok=True)

OPEN_DAY = C.COMPETITOR_OPEN_DAY        # the competitor's actual opening day, 15 March 2024 (C.COMPETITOR_OPEN = the month)

AGE_ORDER = ["18-24", "25-34", "35-44", "45-54", "55+"]   # "Unknown" age (2% of revenue) is left out of charts
AGE_COLORS = {"18-24": "#D1495B", "25-34": "#E09F3E", "35-44": "#2B4C7E",
              "45-54": "#5B8DB8", "55+": "#9AA5B1"}
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold",
                     "axes.titlelocation": "left", "axes.grid": True, "grid.alpha": 0.25,
                     "legend.frameon": False})

def titles(fig, ax, title, subtitle):
    # Bold takeaway on top, then a smaller grey line saying what the chart shows
    fig.suptitle(title, x=0.01, ha="left", fontweight="bold", fontsize=12)
    ax.set_title(subtitle, loc="left", fontweight="normal", fontsize=9.5, color="#555")

lines = load("lines")
visits = load("visits")
customers = load("customers")

p = lines[lines.row_type == C.PURCHASE].copy()           # purchases only (see "Data and definitions")
p["year"], p["mon"] = p.date.dt.year, p.date.dt.month
print(f"{len(p):,} purchase lines, ${p.revenue.sum():,.0f} purchase revenue")

## 1. When did sales change, and for whom?
For each month, revenue by age band compared with the same month a year earlier. The first comparison possible is November 2023, because the data starts in November 2022.

In [ ]:
monthly = p.pivot_table(index="month", columns="age_band", values="revenue", aggfunc="sum")
monthly["All customers"] = monthly.sum(axis=1)
yoy = monthly.pct_change(12, fill_method=None).loc["2023-11-01":]   # change vs the same month 12 months earlier

fig, ax = plt.subplots(figsize=(11, 4.8))
for age in AGE_ORDER:
    hit = age == "18-24"
    ax.plot(yoy.index, yoy[age], color=AGE_COLORS[age], lw=2.6 if hit else 1.3,
            alpha=1 if hit else 0.8, label=age, zorder=3 if hit else 2)
ax.plot(yoy.index, yoy["All customers"], color="black", lw=1.4, ls=":", label="All customers")
ax.axhline(0, color="#333", lw=0.8)
top = ax.get_ylim()[1]
for (d, name), y in zip([(OPEN_DAY, "Competitor opens\n(15 Mar 2024)"), (C.LOYALTY_LAUNCH, "Loyalty\nlaunch")],
                        [top * 0.97, top * 0.72]):
    ax.axvline(d, color="#555", ls="--", lw=0.9)
    ax.text(d, y, " " + name, fontsize=8.5, va="top")
# From Mar 2025, each month is compared with a post-competitor month (a low base), so YoY jumps partly reflect that
ax.axvspan(pd.Timestamp("2025-03-01"), yoy.index.max(), color="#9AA5B1", alpha=0.08)
ax.text(pd.Timestamp("2025-03-15"), ax.get_ylim()[0] * 0.9,
        "Shaded: compared with the weak months right after the competitor\nopened, so these months look better than they really are",
        fontsize=8, color="#555")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set(ylabel="Sales vs. the same month a year earlier", xlabel="")
titles(fig, ax, "After the competitor opened, sales to 18–24 year-olds fell by about half; older shoppers dipped only briefly",
       "Each line is one age group. 0% = same sales as the same month a year earlier; −50% = half as much.")
ax.legend(title="Age group", ncol=6, loc="lower center", bbox_to_anchor=(0.5, -0.3), fontsize=9)
plt.tight_layout(); fig.savefig(OUT / "a1_yoy_by_age.png", bbox_inches="tight"); plt.show()

**What this shows.**
- **Before the competitor:** sales were growing. The store sold 6–13% more each month than a year earlier.
- **From March 2024:** every age group sold less, but 18–24 year-olds (red line) dropped much further, to about half of their previous sales, and stayed there for a whole year.
- **Older shoppers:** back to last year's level by late 2024.

**Watch out for the shaded area.** From March 2025, each month is compared with a weak month right after the competitor opened. Beating a bad month is easy, so the +20% to +60% jumps there overstate the real improvement. That's why later sections compare everything with **2023**, the last normal year.

### Does the drop line up with the opening date?
The competitor opened on **15 March 2024**. If it caused the drop, revenue should be normal until mid-March and then fall. First the monthly view (Nov 2023 – Jun 2024), then a day-level check either side of 15 March.

In [ ]:
yoy_tbl = yoy.loc["2023-11-01":"2024-06-01", AGE_ORDER + ["All customers"]]
yoy_tbl.index = yoy_tbl.index.strftime("%b %Y")
display(yoy_tbl.style.format("{:+.0%}").background_gradient(cmap="RdYlGn", vmin=-0.5, vmax=0.5)
        .set_caption("<b>Revenue vs. same month a year earlier</b>"))

# The first month 18-24 revenue drops more than 20% below the prior year
first_break = yoy.loc[yoy["18-24"] < -0.20].index.min()
print(f"First month 18-24 YoY falls below -20%: {first_break:%B %Y}  (competitor opened {OPEN_DAY:%d %B %Y})")

In [ ]:
# Day-level check: compare the same calendar days in 2024 and 2023, split at the opening day
p["group"] = np.where(p.age_band == "18-24", "18-24", "25+")
periods = {"1–29 Feb": ("02-01", "02-29"), "1–14 Mar (before opening)": ("03-01", "03-14"),
           "15–31 Mar (after opening)": ("03-15", "03-31"), "1–30 Apr": ("04-01", "04-30")}
rows = {}
for name, (start, end) in periods.items():
    r24 = p[(p.date >= f"2024-{start}") & (p.date <= f"2024-{end}")].groupby("group").revenue.sum()
    r23 = p[(p.date >= f"2023-{start}") & (p.date <= f"2023-{end.replace('02-29', '02-28')}")].groupby("group").revenue.sum()
    rows[name] = r24 / r23 - 1
around = pd.DataFrame(rows).T[["18-24", "25+"]]
around.columns = ["Ages 18–24", "Ages 25+"]
(around.style.format("{:+.0%}")
 .set_caption("<b>Revenue, 2024 vs the same days of 2023</b>, either side of the competitor's opening on 15 March"))

**What this shows.** For the whole of March, 18–24 revenue was −25%. That's because **only the second half of March had the competitor**:
- **1–14 March:** 18–24 revenue was normal (+2% vs the same days of 2023), like February (+8%).
- **15–31 March:** it fell **50%** overnight, and stayed −45% in April. Older customers fell too, but much less (−14%).

The break happens on the opening date itself, which is strong evidence the competitor caused it.

## 2. Who was hit hardest?
Revenue in Mar–Oct 2024 compared with Mar–Oct 2023 (see "Comparison window" above), split four ways: age band, product category, home area, and how the customer first found the store (acquisition channel). The "2025 vs 2023" column shows whether each group recovered a year later.

In [ ]:
def window(df, year, m0=3, m1=10):
    # rows from months m0..m1 (default March..October) of one year
    return df[(df.year == year) & df.mon.between(m0, m1)]

def yoy_by(key, m0=3, m1=10):
    base = window(p, 2023, m0, m1).groupby(key).revenue.sum()
    hit = window(p, 2024, m0, m1).groupby(key).revenue.sum()
    later = window(p, 2025, m0, m1).groupby(key).revenue.sum()
    return pd.DataFrame({
        "revenue 2023": base,
        "share of 2023": base / base.sum(),
        "2024 vs 2023": hit / base - 1,
        "lost $ (2024)": hit - base,
        "2025 vs 2023": later / base - 1,
    }).sort_values("2024 vs 2023")

fmt = {"revenue 2023": "${:,.0f}", "share of 2023": "{:.0%}", "2024 vs 2023": "{:+.0%}",
       "lost $ (2024)": "${:,.0f}", "2025 vs 2023": "{:+.0%}"}
tables = {k: yoy_by(k) for k in ["age_band", "category", "home_area", "acquisition_channel"]}
for k, t in tables.items():
    print(f"\n=== {k}: Mar–Oct, each year vs. Mar–Oct 2023 ===")
    display(t.style.format(fmt))

**What this shows.**
- **Age:** 18–24 fell 45%, 25–34 fell 15%, and the other ages fell 3–13%. By 2025 every band except 18–24 was back at or above 2023 (+3% to +22%); 18–24 was still −42%.
- **Category:** leggings fell 30% ($310K), about double any other category (−9% to −15%).
- **Area:** Burnaby (−26%) and Richmond (−25%) fell most; the other neighbourhoods fell 10–17%.
- **Channel:** every acquisition channel fell by a similar amount (−14% to −19%), so the loss wasn't tied to how customers found the store. (The Pilates Campaign channel only started in 2025, so it has no 2023 figure.)

In [ ]:
# Check: Mar–May only, the months with the competitor but before the loyalty program launched
clean = pd.DataFrame({"Mar–Oct": tables["age_band"]["2024 vs 2023"],
                      "Mar–May (before loyalty)": yoy_by("age_band", 3, 5)["2024 vs 2023"]})
display(clean.loc[AGE_ORDER].style.format("{:+.0%}").set_caption("<b>2024 vs 2023 revenue: full window vs before loyalty launched</b>"))

# Check: the first full year after the competitor (Mar 2024–Feb 2025) vs the year before (Mar 2023–Feb 2024)
def year_rev(start, end):
    return p[(p.date >= start) & (p.date < end)].groupby("age_band").revenue.sum()
full = year_rev("2024-03-01", "2025-03-01") / year_rev("2023-03-01", "2024-03-01") - 1
full_total = (p[(p.date >= "2024-03-01") & (p.date < "2025-03-01")].revenue.sum()
              / p[(p.date >= "2023-03-01") & (p.date < "2024-03-01")].revenue.sum() - 1)
print("Full first year vs year before:", ", ".join(f"{a} {full[a]:+.0%}" for a in AGE_ORDER),
      f"| whole store {full_total:+.0%}")

**What this shows.** The result doesn't depend on the window chosen.
- **Before loyalty (Mar–May):** 18–24 was already −40%, so loyalty didn't cause the drop.
- **Full first year (Mar 2024 – Feb 2025):** 18–24 was −43%, almost the same as −45% for Mar–Oct.
- **Store total:** this one *does* depend on the window: −17% for Mar–Oct, −12% for the full year, because older customers were recovering by late 2024. When quoting a store-wide figure, say which window it covers.

In [ ]:
# Is the leggings drop just young customers? Category change inside each age band (Mar–Oct 2024 vs 2023)
a = window(p, 2023).pivot_table(index="age_band", columns="category", values="revenue", aggfunc="sum")
b = window(p, 2024).pivot_table(index="age_band", columns="category", values="revenue", aggfunc="sum")
age_cat = (b / a - 1).loc[AGE_ORDER]
display(age_cat.style.format("{:+.0%}").background_gradient(cmap="RdYlGn", vmin=-0.5, vmax=0.5, axis=None)
        .set_caption("<b>Revenue change by category within each age band</b> (Mar–Oct 2024 vs 2023)"))

lost = (b - a).loc[AGE_ORDER]
print(f"Leggings lost ${-lost['Leggings'].sum():,.0f}; only "
      f"{lost.loc['18-24', 'Leggings'] / lost['Leggings'].sum():.0%} of that loss came from 18-24 customers")
print(f"Leggings fell in every age band: {age_cat['Leggings'].min():+.0%} to {age_cat['Leggings'].max():+.0%} "
      "-> a category-wide hit on a staple, on top of the 18-24 loss")
print(f"18-24 share of all lost revenue: {lost.loc['18-24'].sum() / lost.sum().sum():.0%} "
      f"(they were {tables['age_band'].loc['18-24', 'share of 2023']:.0%} of 2023 revenue)")

**What this shows.**
- **18–24s fell in every category** (−39% to −58%). They didn't switch away from one product; they shopped less across the board.
- **Leggings fell in every age band** (−16% to −58%), and only 35% of the leggings loss came from 18–24s. That fits the case's note that the competitor undercut prices on staple items. Leggings were a separate, store-wide hit on top of losing young customers.
- **18–24s caused 53% of all lost revenue** while making up 19% of 2023 revenue.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, key, title, n_hi in [(axes[0], "category", "By type of product", 1), (axes[1], "home_area", "By where shoppers live", 2)]:
    t = tables[key]["2024 vs 2023"].drop("Unknown", errors="ignore").sort_values()
    colors = ["#D1495B" if i < n_hi else "#9AA5B1" for i in range(len(t))]   # highlight the hardest hit
    ax.barh(t.index, t.values, color=colors)
    for i, v in enumerate(t.values):
        ax.text(v + 0.004, i, f"{v:.0%}", va="center", ha="left", fontsize=8.5, color="white", fontweight="bold")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(1.0, decimals=0))
    ax.set(title=title, xlabel="Change in sales vs. a year earlier")
    ax.invert_yaxis()
fig.suptitle("Leggings lost the most sales (−30%), and shoppers from Burnaby and Richmond cut back the most",
             x=0.01, ha="left", fontweight="bold", fontsize=12)
fig.text(0.01, 0.9, "Sales from March to October 2024 (competitor open) compared with the same months of 2023. "
         "Every bar is negative: everything sold less.", ha="left", fontsize=9.5, color="#555")
plt.tight_layout(rect=(0, 0, 1, 0.93)); fig.savefig(OUT / "a1_hardest_hit_category_area.png", bbox_inches="tight"); plt.show()

## 3. Why did 18–24 revenue fall?
Revenue = **active customers × visits per customer × spend per visit**. Splitting the drop into these three parts shows whether the competitor took whole customers, some of their shopping trips, or made their baskets smaller. Here a "visit" is a transaction that included a purchase.

In [ ]:
v = (visits[visits.has_purchase]
     .merge(customers[["customer_id", "age_band"]], on="customer_id", validate="many_to_one"))
v["year"], v["mon"] = v.date.dt.year, v.date.dt.month

def drivers(year):
    w = window(v, year)
    g = w.groupby("age_band").agg(customers=("customer_id", "nunique"),
                                  visits=("transaction_id", "size"), revenue=("revenue", "sum"))
    g["visits per customer"] = g.visits / g.customers
    g["spend per visit"] = g.revenue / g.visits
    return g.loc[AGE_ORDER, ["customers", "visits per customer", "spend per visit", "revenue"]]

d23, d24, d25 = drivers(2023), drivers(2024), drivers(2025)
change24 = d24 / d23 - 1
change25 = d25 / d23 - 1
display(pd.concat({"Mar–Oct 2023": d23, "2024 vs 2023": change24, "2025 vs 2023": change25}, axis=1)
        .style.format("{:,.2f}").format("{:+.0%}", subset=["2024 vs 2023", "2025 vs 2023"]))

In [ ]:
metrics = ["customers", "visits per customer", "spend per visit"]
fig, ax = plt.subplots(figsize=(9, 3.8))
x = np.arange(len(metrics)); w = 0.36
for i, (lab, ch, col) in enumerate([("2024 (first year with the competitor)", change24, "#D1495B"),
                                    ("2025", change25, "#2B4C7E")]):
    vals = ch.loc["18-24", metrics].values
    ax.bar(x + (i - 0.5) * w, vals, w, color=col, label=lab)
    for xi, val in zip(x + (i - 0.5) * w, vals):
        label = "0%" if abs(val) < 0.005 else f"{val:+.0%}"
        ax.text(xi, val + (0.01 if val >= 0 else -0.01), label, ha="center",
                va="bottom" if val >= 0 else "top", fontsize=9)
ax.axhline(0, color="#333", lw=0.8)
ax.set_xticks(x, ["Number of shoppers", "Shopping trips per shopper", "Spend per trip"])
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set(ylabel="Change vs. 2023 (before the competitor)", ylim=(-0.52, 0.12))
titles(fig, ax, "Fewer 18–24 year-olds came, and they came less often, but they spent the same per trip",
       "18–24 year-olds, March to October each year, compared with the same months of 2023")
ax.legend(loc="lower right")
plt.tight_layout(); fig.savefig(OUT / "a1_18_24_drivers.png", bbox_inches="tight"); plt.show()

**What this shows.**
- **2024:** 27% fewer young shoppers came at all, and those who did came 25% less often. But each trip was worth the same as before ($113).
- **2025:** the young shoppers who stayed came as often as they used to (+4%), but there were 43% fewer of them.

**Why it matters:** the problem isn't what young people buy once they're in the store. It's that **fewer of them come at all**. The fix has to be about winning young shoppers back, not about bigger baskets. The notebook `a3_signups_retention.ipynb` looks at this in detail.

## 4. Did they come back?
Mar–Oct revenue for each year, indexed so that Mar–Oct 2023 = 100 for each age band. 2026 only has data to October, so Mar–Oct is the comparable window.

In [ ]:
rev = pd.DataFrame({y: window(p, y).groupby("age_band").revenue.sum() for y in [2023, 2024, 2025, 2026]}).loc[AGE_ORDER]
idx = rev.div(rev[2023], axis=0) * 100      # index: Mar–Oct 2023 = 100

fig, ax = plt.subplots(figsize=(9, 4))
for age in AGE_ORDER:
    hit = age == "18-24"
    ax.plot(idx.columns.astype(str), idx.loc[age], marker="o", color=AGE_COLORS[age],
            lw=2.6 if hit else 1.4, label=age)
    ax.text(3.08, idx.loc[age, 2026], f"{age}  {idx.loc[age, 2026]:.0f}", va="center", fontsize=8.5,
            color=AGE_COLORS[age], fontweight="bold" if hit else "normal")
ax.axhline(100, color="#333", lw=0.8)
ax.text(-0.15, 101.5, "2023 level (before the competitor)", fontsize=8, color="#555")
ax.set(ylabel="Sales compared with 2023 (2023 = 100)", xlim=(-0.2, 3.6))
titles(fig, ax, "Every age group except 18–24 now buys more than before the competitor",
       "Sales from March to October each year. 100 = the same as 2023; 50 = half as much")
plt.tight_layout(); fig.savefig(OUT / "a1_recovery_index.png", bbox_inches="tight"); plt.show()
idx.round(0)

**What this shows.** Shoppers aged 25 and over now buy 15–32% more than in 2023. 18–24 year-olds buy about **half** as much as in 2023, and that hasn't improved in three years.

**Why it matters:** when leadership says "revenue has recovered", that recovery comes **entirely from shoppers aged 25 and over**. The young shoppers the competitor took never came back, and neither the loyalty program (June 2024) nor free repairs (April 2025) changed that.

## 5. Key numbers

In [ ]:
age = tables["age_band"]; cat = tables["category"]; area = tables["home_area"]
total23 = window(p, 2023).revenue.sum(); total24 = window(p, 2024).revenue.sum()
total25 = window(p, 2025).revenue.sum()
summary = {
    "Competitor opened": f"{OPEN_DAY:%d %B %Y}",
    "18-24 revenue, 1–14 Mar / 15–31 Mar 2024 vs same days 2023": f"{around.iloc[1, 0]:+.0%} / {around.iloc[2, 0]:+.0%}",
    "Store revenue Mar–Oct 2024 vs 2023": f"{total24 / total23 - 1:+.0%} (${total24 - total23:,.0f})",
    "Store revenue Mar–Oct 2025 vs 2023": f"{total25 / total23 - 1:+.0%}",
    "18-24 revenue 2024 vs 2023": f"{age.loc['18-24', '2024 vs 2023']:+.0%}",
    "18-24 revenue 2025 vs 2023": f"{age.loc['18-24', '2025 vs 2023']:+.0%}",
    "18-24 share of all lost revenue (2024)": f"{lost.loc['18-24'].sum() / lost.sum().sum():.0%}",
    "18-24 active customers 2024 / 2025 vs 2023": f"{change24.loc['18-24', 'customers']:+.0%} / {change25.loc['18-24', 'customers']:+.0%}",
    "18-24 spend per visit 2024 vs 2023": f"{change24.loc['18-24', 'spend per visit']:+.0%}".replace("-0%", "0%"),
    "Hardest-hit category": f"{cat.index[0]} {cat.iloc[0]['2024 vs 2023']:+.0%}",
    "Hardest-hit areas": ", ".join(f"{i} {r['2024 vs 2023']:+.0%}" for i, r in area.drop('Unknown').head(2).iterrows()),
    "Outerwear revenue 2025 vs 2023": f"{cat.loc['Outerwear', '2025 vs 2023']:+.0%} overall",
}
# Is the 2025 outerwear dip an age effect? (relevant to the claim that repairs reduce jacket sales)
c25 = window(p, 2025).pivot_table(index="age_band", columns="category", values="revenue", aggfunc="sum")
ow = (c25 / a - 1).loc[AGE_ORDER, "Outerwear"]
summary["Outerwear 2025 vs 2023, by age band"] = ", ".join(f"{k} {v:+.0%}" for k, v in ow.items())
pd.Series(summary, name="value").to_frame()

## Conclusions and caveats
1. **The competitor opened on 15 March 2024, and 18–24 revenue fell 50% that same fortnight**. It hit **18–24 year-olds** hardest: −45% revenue over Mar–Oct, 53% of all lost revenue.
2. **Leggings** were the most-affected category (−30%) across every age, consistent with the competitor undercutting staple items.
3. Young customers **stopped coming** rather than spending less per visit, and **never came back**. The recovery leadership sees is driven by customers aged 25+.
4. **Outerwear** was still 10% below 2023 in 2025, but almost entirely because of 18–24s (−46%). For other ages it was flat or up. So the weak outerwear sales alone are **not evidence** that the repair program (launched Apr 2025) is stealing jacket sales.

**Caveats.** We have no data on the competitor itself (its prices or products), so the link rests on timing, though the drop starting on the opening day makes it strong. The Mar–Oct window includes two pre-competitor weeks (1–14 March), which slightly understates the drop. Results compare periods, so other changes at the same time (the loyalty launch, Jun 2024) could contribute; the Mar–May check suggests they don't drive the 18–24 drop.

---
*Team reference: plan item A-1 (guiding question 1 in the case brief). Findings F1–F5 in `results/a_competitor/findings.md`. Charts saved to `results/a_competitor/`.*